# Cluster-conditioned patch composition

**Role:** Analysis.

Load a trained model and its exact validation cohort, cluster final node embeddings, and sample strictly non-overlapping radius-two patches. Report patch-level marker fractions and coverage, with an optional within-cluster best-predicted subset. The original patch sampling, figures, tables and export workflow are retained. Model loading supports ordinary GIN and FiLM without retraining, and global head features are excluded from local embeddings. Cluster display labels remain ordered by median measured curvature. Accuracy-filtered patches are a selected population, so the all-patch comparison is retained alongside them.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Data and analysis settings

Select the dataset or saved run and the analysis options below. Model-dependent analyses restore the recorded inputs and fitted transformations before computing results.

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
TRAINING_NOTEBOOK = 'gin_depth_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'patch_composition'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
target_transform=selection['transform']
g_val_raw=[selection['raw_graphs'][g.organoid_str].clone() for g in g_val]
# Patch feature columns follow the selected model's marker panel.
for raw,prepared in zip(g_val_raw,g_val):
    raw.x=prepared.x[:,:len(marker_names)].clone()
SAVE_DIR = OUTPUT_DIR  # Destination for the current run or analysis artifacts.
FIGURES_DIR = OUTPUT_DIR/'figures'  # Destination for the generated figures.
TABLES_DIR = OUTPUT_DIR/'tables'  # Destination for exported numerical tables.
FIGURES_DIR.mkdir(exist_ok=True);TABLES_DIR.mkdir(exist_ok=True)

# Inference resources
BATCH_SIZE = 64  # Number of graphs or subgraphs per batch.

# Dataset and targets
TARGET_INDEX_FOR_ANALYSIS = 0  # Target column used for plots and scalar summaries.

# Embedding and clustering parameters
N_CLUSTERS = 6  # Number of embedding clusters to fit.
EMBEDDING_VARIANT = 'local'  # local excludes the head’s appended global features.
RESIDUALIZE_GLOBAL_FOR_CLUSTERING = False  # Regress global-feature associations out of embeddings.
CLUSTERING_METHOD = 'gmm'  # Embedding clustering algorithm: gmm or kmeans.
COVARIANCE_TYPE = 'full'  # GMM covariance parameterization, such as full or diag.
STANDARDIZE_EMBEDDINGS = True  # Standardize hidden coordinates before clustering.
PCA_DIM = None  # PCA dimensions used before clustering; None keeps all.
CLUSTER_SEED = 42  # Random seed for reproducible sampling or initialization.

# Patch sampling
PATCH_RADIUS = 2  # Graph-hop radius of each sampled patch.
PATCH_SEED = 42  # Random seed for reproducible sampling or initialization.
MIN_MARKER_HOP_COVERAGE = 10  # Desired patch support for each marker/hop combination.
COVERAGE_CANDIDATE_POOL = 2000  # Candidate patches considered during coverage optimization.
MAX_STAGE2_PATCHES = None  # Cap additional coverage patches; None uses no cap.

# Patch quality selection
ACCURATE_PATCH_FRACTION = .5  # Best-predicted fraction retained within each cluster.
ACCURATE_PATCH_MIN_PER_CLUSTER = 1  # Minimum retained patches per cluster after filtering.

# Dataset and targets
TARGET_INDEX_FOR_ANALYSIS = 0  # Target column used for plots and scalar summaries.

# Inference resources
BATCH_SIZE = 128  # Number of graphs or subgraphs per batch.

# Embedding and clustering parameters
N_CLUSTERS = 9  # Number of embedding clusters to fit.
EMBEDDING_VARIANT = 'local'  # local excludes the head’s appended global features.
RESIDUALIZE_GLOBAL_FOR_CLUSTERING = False  # Regress global-feature associations out of embeddings.
CLUSTERING_METHOD = 'gmm'  # Embedding clustering algorithm: gmm or kmeans.
COVARIANCE_TYPE = 'full'  # GMM covariance parameterization, such as full or diag.
STANDARDIZE_EMBEDDINGS = True  # Standardize hidden coordinates before clustering.
PCA_DIM = 32  # PCA dimensions used before clustering; None keeps all.
CLUSTER_SEED = 0  # Random seed for reproducible sampling or initialization.

# Patch sampling
PATCH_RADIUS = 2  # Graph-hop radius of each sampled patch.
MIN_MARKER_HOP_COVERAGE = 100  # Desired patch support for each marker/hop combination.
COVERAGE_CANDIDATE_POOL = 2000  # Candidate patches considered during coverage optimization.
MAX_STAGE2_PATCHES = None  # Cap additional coverage patches; None uses no cap.
PATCH_SEED = 0  # Random seed for reproducible sampling or initialization.

# Patch quality selection
ACCURATE_PATCH_FRACTION = 0.5  # Best-predicted fraction retained within each cluster.
ACCURATE_PATCH_MIN_PER_CLUSTER = 1  # Minimum retained patches per cluster after filtering.

In [ ]:
import copy
import json
import pickle
import random
import sys
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

def save_figure(fig, name, *, dpi=300):
    for suffix in ('pdf', 'png'):
        fig.savefig(FIGURES_DIR / f'{name}.{suffix}', dpi=dpi, bbox_inches='tight')

def select_target_column(values, target_index=0):
    arr = np.asarray(values)
    if arr.ndim == 1:
        return arr
    return arr[:, int(target_index)]

def jsonable(value):
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [jsonable(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, (torch.dtype, np.dtype)):
        return str(value)
    return value

def split_flat_by_graph(values, graphs):
    values = np.asarray(values)
    output = []
    offset = 0
    for graph in graphs:
        n_nodes = int(graph.x.shape[0])
        output.append(values[offset:offset + n_nodes].copy())
        offset += n_nodes
    if offset != len(values):
        raise ValueError(f'Consumed {offset} values but received {len(values)}.')
    return output

from src.data.filters import filter_graphs_by_blacklist, filter_graphs_by_marker_diversity, filter_graphs_by_metadata, filter_graphs_by_numeric_metadata, filter_graphs_by_sphericity, load_graph_blacklist_from_dir
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import add_log_metadata_features, attach_metadata_to_graphs, fill_missing_metadata_for_group, infer_global_dim, load_aux_metadata_for_dir, load_marker_names_from_dir, promote_metadata_to_graph_tensors, snapshot_graph_metadata, strip_graph_metadata
from src.data.preprocessing import interpolate_target_outliers_from_neighbors
from src.data.splits import graph_metadata_key, train_val_split_graphs
from src.data.target_transforms import AsinhStandardizeTransform, ChainedTargetTransform, GlobalBaselineResidualTransform, standardize_graph_global_features
from src.models.gnn import GINCurvature
from src.training.losses import WeightedLossTerm, edge_loss_term
from src.artifacts.bundle import save_bundle, graph_membership

## 3. Cluster Validation Node Embeddings

Extract final local node embeddings, fit the configured clustering method and order displayed labels by median measured curvature. The following patch statistics use these same cluster assignments.

In [ ]:
from src.analysis.embeddings.clustering import run_embedding_clustering

extraction, clustering_result, cluster_summary_raw = run_embedding_clustering(
    graphs=g_val,
    model=model,
    batch_size=BATCH_SIZE,
    center_only=False,
    embedding_variant=EMBEDDING_VARIANT,
    residualize_global=RESIDUALIZE_GLOBAL_FOR_CLUSTERING,
    clustering=CLUSTERING_METHOD,
    n_clusters=N_CLUSTERS,
    covariance_type=COVARIANCE_TYPE,
    standardize=STANDARDIZE_EMBEDDINGS,
    pca_dim=PCA_DIM,
    seed=CLUSTER_SEED,
    marker_names=marker_names,
)

y_true, y_pred, log_var = target_transform.inverse_distribution(
    extraction.y_true,
    extraction.y_pred,
    log_var=extraction.log_var,
    graphs=g_val,
)
offsets = np.concatenate([np.asarray(selection['baseline_offsets'][g.organoid_str]).reshape(-1) for g in g_val])
y_true = select_target_column(y_true, TARGET_INDEX_FOR_ANALYSIS) + offsets
y_pred = select_target_column(y_pred, TARGET_INDEX_FOR_ANALYSIS) + offsets
labels = np.asarray(clustering_result.labels, dtype=int)

# Relabel clusters by median true curvature for stable, interpretable display order.
unique_labels = np.sort(np.unique(labels))
cluster_medians = {
    int(cluster): float(np.nanmedian(y_true[labels == cluster]))
    for cluster in unique_labels
}
old_to_new = {
    old: new
    for new, old in enumerate(sorted(unique_labels, key=lambda x: cluster_medians[int(x)]))
}
labels = np.asarray([old_to_new[int(label)] for label in labels], dtype=int)
clustering_result.labels = labels

labels_by_graph = split_flat_by_graph(labels, g_val_raw)
y_true_by_graph = split_flat_by_graph(y_true, g_val_raw)
y_pred_by_graph = split_flat_by_graph(y_pred, g_val_raw)

cluster_summary_df = pd.DataFrame([{
    "cluster": int(cluster),
    "n_validation_nodes": int(np.sum(labels == cluster)),
    "median_true_curvature": float(np.nanmedian(y_true[labels == cluster])),
    "mean_true_curvature": float(np.nanmean(y_true[labels == cluster])),
    "mean_predicted_curvature": float(np.nanmean(y_pred[labels == cluster])),
    "median_absolute_error": float(
        np.nanmedian(np.abs(y_pred[labels == cluster] - y_true[labels == cluster]))
    ),
} for cluster in np.sort(np.unique(labels))])
display(cluster_summary_df)


## 4. Sample Strictly Non-Overlapping Radius-2 Patches

Construct radius-two candidates and select patches with no shared nodes. Coverage targets favor representation of rare marker/hop combinations; the resulting patch sample is not a uniform census.

In [ ]:
from scripts.export_patch_composition_tables import (
    build_adjacency,
    graph_identity,
    original_node_id,
)

def exact_hop_rings_from_adjacency(adjacency, center, max_hops):
    visited = {int(center)}
    frontier = {int(center)}
    rings = [[int(center)]]
    for _ in range(int(max_hops)):
        next_frontier = set()
        for node in frontier:
            next_frontier.update(adjacency[node])
        next_frontier.difference_update(visited)
        rings.append(sorted(next_frontier))
        visited.update(next_frontier)
        frontier = next_frontier
    return rings

def build_global_patch_candidates(graphs, marker_names, *, radius):
    n_features = (int(radius) + 1) * len(marker_names)
    features = []
    graph_indices = []
    centers = []
    patch_nodes = []
    exact_rings = []
    conflict_centers_local = []
    graph_offsets = []
    offset = 0

    for graph_idx, graph in enumerate(graphs):
        graph_offsets.append(offset)
        adjacency = build_adjacency(graph)
        x_positive = graph.x.detach().cpu().numpy() > 0.5
        n_nodes = int(graph.x.shape[0])
        for center in range(n_nodes):
            rings = exact_hop_rings_from_adjacency(
                adjacency,
                center,
                2 * int(radius),
            )
            patch_ring_nodes = rings[: int(radius) + 1]
            patch = np.asarray(
                sorted({node for ring in patch_ring_nodes for node in ring}),
                dtype=np.int64,
            )
            conflict = np.asarray(
                sorted({node for ring in rings for node in ring}),
                dtype=np.int64,
            )
            feature_matrix = np.zeros(
                (int(radius) + 1, len(marker_names)),
                dtype=bool,
            )
            for hop, ring in enumerate(patch_ring_nodes):
                if ring:
                    feature_matrix[hop] = x_positive[
                        np.asarray(ring, dtype=np.int64)
                    ].any(axis=0)

            graph_indices.append(graph_idx)
            centers.append(center)
            patch_nodes.append(patch)
            exact_rings.append([
                np.asarray(ring, dtype=np.int64)
                for ring in patch_ring_nodes
            ])
            conflict_centers_local.append(conflict)
            features.append(feature_matrix.reshape(n_features))
        offset += n_nodes

    graph_offsets = np.asarray(graph_offsets, dtype=np.int64)
    graph_indices = np.asarray(graph_indices, dtype=np.int64)
    centers = np.asarray(centers, dtype=np.int64)
    features = np.asarray(features, dtype=bool)
    conflict_candidate_ids = [
        graph_offsets[graph_idx] + local_centers
        for graph_idx, local_centers in zip(
            graph_indices,
            conflict_centers_local,
        )
    ]
    return {
        "features": features,
        "graph_indices": graph_indices,
        "centers": centers,
        "patch_nodes": patch_nodes,
        "exact_rings": exact_rings,
        "conflict_candidate_ids": conflict_candidate_ids,
    }

def sample_two_stage_non_overlapping_patches(
    graphs,
    marker_names,
    *,
    radius,
    min_marker_hop_coverage,
    candidate_pool_size,
    max_stage2_patches,
    seed,
):
    if max_stage2_patches is not None and int(max_stage2_patches) < 0:
        raise ValueError("max_stage2_patches must be non-negative or None.")

    candidates = build_global_patch_candidates(
        graphs,
        marker_names,
        radius=radius,
    )
    features = candidates["features"]
    n_candidates, n_features = features.shape
    rng = np.random.default_rng(seed)
    available = np.ones(n_candidates, dtype=bool)
    coverage = np.zeros(n_features, dtype=np.int64)
    possible = features.sum(axis=0).astype(np.int64)
    requested = np.full(
        n_features,
        int(min_marker_hop_coverage),
        dtype=np.int64,
    )
    selected_ids = []
    selected_stages = []
    feature_candidates = [
        np.flatnonzero(features[:, feature_idx])
        for feature_idx in range(n_features)
    ]

    def select_candidate(candidate_id, stage):
        candidate_id = int(candidate_id)
        selected_ids.append(candidate_id)
        selected_stages.append(stage)
        coverage[:] += features[candidate_id]
        available[candidates["conflict_candidate_ids"][candidate_id]] = False

    # Stage 1: target overall support for every marker at every exact hop.
    while np.any(coverage < requested):
        unsatisfied = np.flatnonzero(coverage < requested)
        relative_coverage = coverage[unsatisfied] / np.maximum(
            requested[unsatisfied],
            1,
        )
        priority_order = unsatisfied[
            np.lexsort((possible[unsatisfied], relative_coverage))
        ]
        chosen = None
        for feature_idx in priority_order:
            eligible = feature_candidates[int(feature_idx)]
            eligible = eligible[available[eligible]]
            if len(eligible) == 0:
                continue
            if candidate_pool_size is not None and len(eligible) > int(candidate_pool_size):
                eligible = rng.choice(
                    eligible,
                    size=int(candidate_pool_size),
                    replace=False,
                )

            unmet = coverage < requested
            rarity_weights = unmet / np.maximum(possible, 1)
            scores = features[eligible].astype(float) @ rarity_weights
            patch_sizes = np.asarray(
                [len(candidates["patch_nodes"][int(idx)]) for idx in eligible],
                dtype=float,
            )
            scores += 1e-6 / np.maximum(patch_sizes, 1)
            best = np.flatnonzero(scores == scores.max())
            chosen = int(eligible[int(rng.choice(best))])
            break
        if chosen is None:
            break
        select_candidate(chosen, "minimum_coverage")

    stage1_coverage = coverage.copy()

    # Stage 2: fill to a maximal disjoint packing, prioritizing features that
    # remain uncommon in the stage-1 sample.
    feature_weights = 1.0 / (stage1_coverage + 1.0)
    stage2_scores = features.astype(float) @ feature_weights
    patch_sizes = np.asarray(
        [len(nodes) for nodes in candidates["patch_nodes"]],
        dtype=float,
    )
    stage2_scores += 1e-3 / np.maximum(patch_sizes, 1)
    stage2_scores += rng.uniform(0, 1e-9, size=n_candidates)
    n_stage2_selected = 0
    for candidate_id in np.argsort(-stage2_scores):
        if (
            max_stage2_patches is not None
            and n_stage2_selected >= int(max_stage2_patches)
        ):
            break
        if available[int(candidate_id)]:
            select_candidate(int(candidate_id), "coverage_maximization")
            n_stage2_selected += 1

    selected = []
    for candidate_id, stage in zip(selected_ids, selected_stages):
        selected.append({
            "candidate_id": int(candidate_id),
            "graph_index": int(candidates["graph_indices"][candidate_id]),
            "center": int(candidates["centers"][candidate_id]),
            "patch_nodes": candidates["patch_nodes"][candidate_id],
            "exact_rings": candidates["exact_rings"][candidate_id],
            "features": features[candidate_id].reshape(
                int(radius) + 1,
                len(marker_names),
            ),
            "sampling_stage": stage,
        })

    rows = []
    final_coverage = coverage.copy()
    for hop in range(int(radius) + 1):
        for marker_idx, marker in enumerate(marker_names):
            feature_idx = hop * len(marker_names) + marker_idx
            rows.append({
                "hop": hop,
                "marker_index": marker_idx,
                "marker": marker,
                "requested_minimum": int(requested[feature_idx]),
                "candidate_centers_available": int(possible[feature_idx]),
                "stage1_selected_patches": int(stage1_coverage[feature_idx]),
                "final_selected_patches": int(final_coverage[feature_idx]),
                "minimum_achieved": bool(
                    stage1_coverage[feature_idx] >= requested[feature_idx]
                ),
            })
    return selected, pd.DataFrame(rows), candidates

def verify_global_patch_disjointness(selected_patches):
    used_by_graph = {}
    for patch in selected_patches:
        graph_idx = int(patch["graph_index"])
        used = used_by_graph.setdefault(graph_idx, set())
        node_set = set(int(node) for node in patch["patch_nodes"])
        if used.intersection(node_set):
            return False
        used.update(node_set)
    return True


In [ ]:
selected_patches, marker_hop_coverage_df, patch_candidates = (
    sample_two_stage_non_overlapping_patches(
        g_val_raw,
        marker_names,
        radius=PATCH_RADIUS,
        min_marker_hop_coverage=MIN_MARKER_HOP_COVERAGE,
        candidate_pool_size=COVERAGE_CANDIDATE_POOL,
        max_stage2_patches=MAX_STAGE2_PATCHES,
        seed=PATCH_SEED,
    )
)
if not verify_global_patch_disjointness(selected_patches):
    raise RuntimeError("The two-stage sampler produced overlapping patches.")

patches_by_graph = {graph_idx: [] for graph_idx in range(len(g_val_raw))}
for patch in selected_patches:
    patches_by_graph[patch["graph_index"]].append(patch)

patch_rows = []
organoid_sampling_rows = []
for graph_idx, graph in enumerate(g_val_raw):
    graph_patches = sorted(
        patches_by_graph[graph_idx],
        key=lambda patch: patch["center"],
    )
    x = graph.x.detach().cpu().numpy()
    identity = graph_identity(graph)
    n_sampled_cells = sum(len(patch["patch_nodes"]) for patch in graph_patches)
    organoid_sampling_rows.append({
        "graph_index": graph_idx,
        **identity,
        "n_organoid_cells": int(graph.x.shape[0]),
        "n_patches": len(graph_patches),
        "n_minimum_coverage_patches": sum(
            patch["sampling_stage"] == "minimum_coverage"
            for patch in graph_patches
        ),
        "n_coverage_maximization_patches": sum(
            patch["sampling_stage"] == "coverage_maximization"
            for patch in graph_patches
        ),
        "n_cells_in_sampled_patches": n_sampled_cells,
        "sampled_cell_fraction": (
            n_sampled_cells / int(graph.x.shape[0])
            if int(graph.x.shape[0]) > 0
            else np.nan
        ),
        "patches_are_node_disjoint": True,
    })

    for patch_index, patch in enumerate(graph_patches):
        center = patch["center"]
        patch_nodes = patch["patch_nodes"]
        patch_x = x[patch_nodes]
        row = {
            "graph_index": graph_idx,
            **identity,
            "patch_index": patch_index,
            "candidate_id": patch["candidate_id"],
            "sampling_stage": patch["sampling_stage"],
            "center_node_id": center,
            "center_node_original_id": original_node_id(graph, center),
            "patch_radius": PATCH_RADIUS,
            "patch_node_ids": json.dumps([int(node) for node in patch_nodes]),
            "n_cells_patch": len(patch_nodes),
            "cluster": int(labels_by_graph[graph_idx][center]),
            "center_true_curvature": float(y_true_by_graph[graph_idx][center]),
            "center_predicted_curvature": float(y_pred_by_graph[graph_idx][center]),
        }
        row["center_absolute_error"] = abs(
            row["center_predicted_curvature"] - row["center_true_curvature"]
        )
        for marker_idx, marker in enumerate(marker_names):
            positive = patch_x[:, marker_idx] > 0.5
            row[f"n_marker_{marker_idx}_positive"] = int(positive.sum())
            row[f"frac_marker_{marker_idx}_positive"] = float(positive.mean())
            row[f"center_marker_{marker_idx}_positive"] = int(
                x[center, marker_idx] > 0.5
            )
            for hop in range(PATCH_RADIUS + 1):
                row[f"hop_{hop}_marker_{marker_idx}_present"] = int(
                    patch["features"][hop, marker_idx]
                )
        patch_rows.append(row)

patch_df = pd.DataFrame(patch_rows)
organoid_sampling_df = pd.DataFrame(organoid_sampling_rows)
print(
    f"Sampled {len(patch_df):,} non-overlapping patches: "
    f"{sum(patch['sampling_stage'] == 'minimum_coverage' for patch in selected_patches):,} "
    "for minimum coverage and "
    f"{sum(patch['sampling_stage'] == 'coverage_maximization' for patch in selected_patches):,} "
    "during coverage maximization."
)
display(marker_hop_coverage_df)
display(organoid_sampling_df.describe(include="all"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].hist(organoid_sampling_df["n_patches"], bins=20, color="#4C78A8")
axes[0].set_xlabel("non-overlapping patches per organoid")
axes[0].set_ylabel("organoids")
axes[0].set_title("Patch count")

axes[1].hist(patch_df["n_cells_patch"], bins=20, color="#59A14F")
axes[1].set_xlabel("cells per radius-2 patch")
axes[1].set_ylabel("patches")
axes[1].set_title("Patch size")
fig.tight_layout()
save_figure(fig, "patch_sampling_distributions")
plt.show()

coverage_table = (
    marker_hop_coverage_df.pivot(
        index="marker",
        columns="hop",
        values="final_selected_patches",
    )
    .reindex(index=marker_names, columns=range(PATCH_RADIUS + 1))
)
stage1_table = (
    marker_hop_coverage_df.pivot(
        index="marker",
        columns="hop",
        values="stage1_selected_patches",
    )
    .reindex(index=coverage_table.index, columns=coverage_table.columns)
)

coverage_vmax = float(max(stage1_table.to_numpy().max(), coverage_table.to_numpy().max()))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 5.0), sharey=True)
images = []
for ax, table, title in [
    (axes[0], stage1_table, "After minimum-coverage stage"),
    (axes[1], coverage_table, "Final two-stage sample"),
]:
    image = ax.imshow(
        table.to_numpy(float),
        aspect="auto",
        cmap="viridis",
        vmin=0,
        vmax=coverage_vmax,
    )
    images.append(image)
    ax.set_xticks(range(PATCH_RADIUS + 1))
    ax.set_xticklabels([f"hop {hop}" for hop in range(PATCH_RADIUS + 1)])
    ax.set_yticks(range(len(marker_names)))
    ax.set_yticklabels(marker_names)
    ax.set_xlabel("exact marker distance from center")
    ax.set_title(title)
    for row in range(table.shape[0]):
        for col in range(table.shape[1]):
            value = table.iloc[row, col]
            ax.text(
                col,
                row,
                f"{int(value)}",
                ha="center",
                va="center",
                color="white" if value < table.to_numpy().max() * 0.45 else "black",
                fontsize=8,
            )
axes[0].set_ylabel("marker")
fig.suptitle(
    f"Selected-patch marker coverage by exact hop "
    f"(requested minimum = {MIN_MARKER_HOP_COVERAGE})"
)
fig.subplots_adjust(left=0.13, right=0.86, bottom=0.14, top=0.86, wspace=0.18)
colorbar_ax = fig.add_axes([0.89, 0.19, 0.02, 0.60])
colorbar = fig.colorbar(images[-1], cax=colorbar_ax)
colorbar.set_label("selected patches containing marker")
save_figure(fig, "marker_hop_sampling_coverage")
plt.show()


## 5. Marker Statistics By Center-Cell Cluster

Compute marker composition and supporting counts for the selected population. Explicitly distinguish cell counts, patch counts and organoid counts when interpreting these summaries.

In [ ]:
def patch_marker_long_table(patches, stage):
    rows = []
    for marker_idx, marker in enumerate(marker_names):
        columns = [
            "graph_index",
            "organoid_str",
            "patch_index",
            "cluster",
            "center_absolute_error",
            "n_cells_patch",
            f"n_marker_{marker_idx}_positive",
            f"frac_marker_{marker_idx}_positive",
        ]
        marker_df = patches[columns].copy()
        marker_df = marker_df.rename(columns={
            f"n_marker_{marker_idx}_positive": "n_positive",
            f"frac_marker_{marker_idx}_positive": "fraction_positive",
        })
        marker_df["marker_index"] = marker_idx
        marker_df["marker"] = marker
        marker_df["stage"] = stage
        rows.append(marker_df)
    return pd.concat(rows, ignore_index=True)

def summarize_patch_markers(long_df):
    summary = (
        long_df.groupby(["stage", "cluster", "marker_index", "marker"], as_index=False)
        .agg(
            n_patches=("fraction_positive", "size"),
            mean_fraction_positive=("fraction_positive", "mean"),
            std_fraction_positive=("fraction_positive", "std"),
            median_fraction_positive=("fraction_positive", "median"),
            q25_fraction_positive=("fraction_positive", lambda x: x.quantile(0.25)),
            q75_fraction_positive=("fraction_positive", lambda x: x.quantile(0.75)),
            mean_positive_cells=("n_positive", "mean"),
            mean_patch_size=("n_cells_patch", "mean"),
        )
    )
    summary["std_fraction_positive"] = summary["std_fraction_positive"].fillna(0.0)
    summary["sem_fraction_positive"] = (
        summary["std_fraction_positive"]
        / np.sqrt(summary["n_patches"].clip(lower=1))
    )
    return summary

patch_marker_all_df = patch_marker_long_table(patch_df, "all_patches")
marker_summary_all_df = summarize_patch_markers(patch_marker_all_df)
display(marker_summary_all_df.head(12))


In [ ]:
def plot_marker_fraction_heatmaps(
    all_summary,
    accurate_summary,
    *,
    filename,
):
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True)
    stages = [
        (all_summary, "All non-overlapping patches"),
        (accurate_summary, "Best-predicted 50% within each cluster"),
    ]
    image = None
    for ax, (summary_df, title) in zip(axes, stages):
        table = (
            summary_df.pivot(
                index="cluster",
                columns="marker",
                values="mean_fraction_positive",
            )
            .reindex(columns=marker_names)
            .sort_index()
        )
        sem_table = (
            summary_df.pivot(
                index="cluster",
                columns="marker",
                values="sem_fraction_positive",
            )
            .reindex(index=table.index, columns=table.columns)
        )
        image = ax.imshow(table.to_numpy(float), aspect="auto", vmin=0, vmax=1, cmap="viridis")
        ax.set_xticks(np.arange(len(marker_names)))
        ax.set_xticklabels(marker_names, rotation=45, ha="right")
        ax.set_yticks(np.arange(len(table.index)))
        ax.set_yticklabels([f"C{cluster}" for cluster in table.index])
        ax.set_xlabel("marker")
        ax.set_title(title)
        for row in range(table.shape[0]):
            for col in range(table.shape[1]):
                mean = table.iloc[row, col]
                sem = sem_table.iloc[row, col]
                if np.isfinite(mean):
                    ax.text(
                        col,
                        row,
                        f"{mean:.2f}\n±{sem:.2f}",
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="white" if mean < 0.35 else "black",
                    )
    axes[0].set_ylabel("center-cell embedding cluster")
    fig.suptitle("Patch marker composition by center-cell embedding cluster")
    fig.subplots_adjust(left=0.08, right=0.88, bottom=0.18, top=0.88, wspace=0.18)
    colorbar_ax = fig.add_axes([0.91, 0.20, 0.018, 0.62])
    colorbar = fig.colorbar(image, cax=colorbar_ax)
    colorbar.set_label("mean patch fraction positive")
    save_figure(fig, filename)
    plt.show()
    return fig, axes

def plot_patch_marker_distributions(long_df, title, filename):
    n_markers = len(marker_names)
    n_cols = min(4, n_markers)
    n_rows = int(np.ceil(n_markers / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.0 * n_cols, 3.1 * n_rows),
        sharey=True,
        squeeze=False,
    )
    axes = axes.ravel()
    clusters = np.sort(long_df["cluster"].unique())
    for ax, marker in zip(axes, marker_names):
        marker_df = long_df[long_df["marker"] == marker]
        values = [
            marker_df.loc[marker_df["cluster"] == cluster, "fraction_positive"].to_numpy(float)
            for cluster in clusters
        ]
        boxplot = ax.boxplot(values, showfliers=False, patch_artist=True)
        for box in boxplot["boxes"]:
            box.set_facecolor("#4C78A8")
        ax.set_title(marker)
        ax.set_xticks(np.arange(1, len(clusters) + 1))
        ax.set_xticklabels([f"C{cluster}" for cluster in clusters], rotation=45)
        ax.set_ylim(-0.03, 1.03)
        ax.grid(axis="y", alpha=0.2)
    for ax in axes[n_markers:]:
        ax.set_visible(False)
    for row in range(n_rows):
        axes[row * n_cols].set_ylabel("patch fraction positive")
    fig.suptitle(title)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save_figure(fig, filename)
    plt.show()
    return fig, axes

def plot_patch_curvature_boxplots(all_patches, accurate_patches, filename):
    from src.plotting.cluster_plots import plot_cluster_boxplots

    cluster_order = np.sort(all_patches["cluster"].unique())
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14.0, 4.8),
        sharex=True,
        sharey=True,
    )
    panels = [
        (all_patches, "All non-overlapping patches"),
        (accurate_patches, "Best-predicted 50% within each cluster"),
    ]
    for ax, (patches, title) in zip(axes, panels):
        plot_cluster_boxplots(
            [
                patches["center_true_curvature"].to_numpy(float),
                patches["center_predicted_curvature"].to_numpy(float),
            ],
            patches["cluster"].to_numpy(int),
            data_labels=["true", "predicted"],
            colors=["lightblue", "lightgreen"],
            cluster_order=cluster_order,
            show_outliers=False,
            ylabel="center-cell curvature",
            title=title,
            xlabel="center-cell embedding cluster",
            ax=ax,
        )
    fig.suptitle("True and predicted curvature of sampled patch centers")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    save_figure(fig, filename)
    plt.show()
    return fig, axes

def marker_fraction_change_table(all_summary, accurate_summary):
    key_columns = ["cluster", "marker_index", "marker"]
    before = all_summary[
        key_columns + ["mean_fraction_positive", "sem_fraction_positive"]
    ].rename(columns={
        "mean_fraction_positive": "mean_fraction_before",
        "sem_fraction_positive": "sem_fraction_before",
    })
    after = accurate_summary[
        key_columns + ["mean_fraction_positive", "sem_fraction_positive"]
    ].rename(columns={
        "mean_fraction_positive": "mean_fraction_after",
        "sem_fraction_positive": "sem_fraction_after",
    })
    change = before.merge(after, on=key_columns, how="outer", validate="one_to_one")
    change["fraction_change_after_minus_before"] = (
        change["mean_fraction_after"] - change["mean_fraction_before"]
    )
    return change

def plot_marker_fraction_changes(change_df, filename):
    table = (
        change_df.pivot(
            index="cluster",
            columns="marker",
            values="fraction_change_after_minus_before",
        )
        .reindex(columns=marker_names)
        .sort_index()
    )
    values = table.to_numpy(float)
    finite = np.abs(values[np.isfinite(values)])
    limit = float(finite.max()) if finite.size else 1.0
    if limit == 0:
        limit = 1.0

    fig, ax = plt.subplots(figsize=(8.6, 5.2))
    image = ax.imshow(
        values,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-limit,
        vmax=limit,
    )
    ax.set_xticks(np.arange(len(marker_names)))
    ax.set_xticklabels(marker_names, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(table.index)))
    ax.set_yticklabels([f"C{cluster}" for cluster in table.index])
    ax.set_xlabel("marker")
    ax.set_ylabel("center-cell embedding cluster")
    ax.set_title("Marker composition change after accuracy filtering")
    for row in range(table.shape[0]):
        for col in range(table.shape[1]):
            value = values[row, col]
            if np.isfinite(value):
                ax.text(
                    col,
                    row,
                    f"{value:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if abs(value) > 0.55 * limit else "black",
                )
    fig.subplots_adjust(left=0.12, right=0.84, bottom=0.20, top=0.88)
    colorbar_ax = fig.add_axes([0.88, 0.20, 0.022, 0.62])
    colorbar = fig.colorbar(image, cax=colorbar_ax)
    colorbar.set_label("fraction after filtering - fraction before filtering")
    save_figure(fig, filename)
    plt.show()
    return fig, ax


## Keep the best-predicted fraction within each cluster

Rank patches by absolute center-cell prediction error and retain the configured fraction separately within each curvature-ordered cluster. Compare these selected patches with the complete sample to reveal selection effects on marker content.

In [ ]:
def best_patch_mask_by_cluster(
    patches,
    *,
    fraction,
    min_per_cluster=1,
):
    if not (0 < fraction <= 1):
        raise ValueError("fraction must lie in (0, 1].")
    keep = np.zeros(len(patches), dtype=bool)
    rows = []
    for cluster in np.sort(patches["cluster"].unique()):
        positions = np.flatnonzero(patches["cluster"].to_numpy() == cluster)
        errors = patches.iloc[positions]["center_absolute_error"].to_numpy(float)
        finite_positions = positions[np.isfinite(errors)]
        n_available = len(finite_positions)
        n_keep = min(
            n_available,
            max(int(min_per_cluster), int(np.ceil(fraction * n_available))),
        )
        if n_keep:
            ranked = finite_positions[
                np.argsort(
                    patches.iloc[finite_positions]["center_absolute_error"].to_numpy(float),
                    kind="mergesort",
                )
            ]
            chosen = ranked[:n_keep]
            keep[chosen] = True
            threshold = float(
                patches.iloc[chosen]["center_absolute_error"].max()
            )
        else:
            threshold = np.nan
        rows.append({
            "cluster": int(cluster),
            "n_available_patches": n_available,
            "n_kept_patches": n_keep,
            "kept_fraction": n_keep / n_available if n_available else np.nan,
            "absolute_error_threshold": threshold,
            "median_absolute_error_all": float(np.nanmedian(errors)),
            "median_absolute_error_kept": (
                float(np.nanmedian(patches.loc[keep & (patches["cluster"] == cluster), "center_absolute_error"]))
                if n_keep
                else np.nan
            ),
        })
    return keep, pd.DataFrame(rows)

accurate_patch_mask, accuracy_filter_summary_df = best_patch_mask_by_cluster(
    patch_df,
    fraction=ACCURATE_PATCH_FRACTION,
    min_per_cluster=ACCURATE_PATCH_MIN_PER_CLUSTER,
)
accurate_patch_df = patch_df.loc[accurate_patch_mask].copy()
patch_marker_accurate_df = patch_marker_long_table(
    accurate_patch_df,
    "best_predicted_half",
)
marker_summary_accurate_df = summarize_patch_markers(patch_marker_accurate_df)

print(
    f"Retained {len(accurate_patch_df):,}/{len(patch_df):,} patches "
    f"({len(accurate_patch_df) / len(patch_df):.1%})."
)
display(accuracy_filter_summary_df)


In [ ]:
marker_fraction_change_df = marker_fraction_change_table(
    marker_summary_all_df,
    marker_summary_accurate_df,
)

plot_marker_fraction_heatmaps(
    marker_summary_all_df,
    marker_summary_accurate_df,
    filename="patch_marker_fraction_heatmaps_all_vs_accurate",
)
plot_patch_marker_distributions(
    patch_marker_all_df,
    "Patch marker fractions | all non-overlapping patches",
    "patch_marker_distributions_all",
)
plot_patch_marker_distributions(
    patch_marker_accurate_df,
    "Patch marker fractions | best-predicted 50% within each cluster",
    "patch_marker_distributions_accurate_half",
)

plot_patch_curvature_boxplots(
    patch_df,
    accurate_patch_df,
    filename="patch_center_curvature_boxplots_all_vs_accurate",
)
plot_marker_fraction_changes(
    marker_fraction_change_df,
    filename="patch_marker_fraction_changes_after_filtering",
)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
clusters = accuracy_filter_summary_df["cluster"].to_numpy(int)
width = 0.38
ax.bar(
    clusters - width / 2,
    accuracy_filter_summary_df["n_available_patches"],
    width=width,
    label="all patches",
    color="#B8C4CE",
)
ax.bar(
    clusters + width / 2,
    accuracy_filter_summary_df["n_kept_patches"],
    width=width,
    label="best-predicted half",
    color="#4C78A8",
)
ax.set_xticks(clusters)
ax.set_xticklabels([f"C{cluster}" for cluster in clusters])
ax.set_xlabel("center-cell embedding cluster")
ax.set_ylabel("sampled patches")
ax.set_title("Patch counts before and after within-cluster accuracy filtering")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "patch_counts_by_cluster")
plt.show()


## 7. Export Filtered Patch Composition Table

Write the current analysis tables and provenance to its output directory so plots can be reproduced independently.

In [ ]:
from scripts.export_patch_composition_tables import (
    metadata_value,
    slugify,
    target_vector,
)

def build_filtered_patch_export_table(filtered_patches, graphs, marker_names):
    """Create an export-script-compatible table for retained validation patches."""
    marker_slugs = [slugify(marker) for marker in marker_names]
    rows = []

    for patch in filtered_patches.itertuples(index=False):
        graph = graphs[int(patch.graph_index)]
        patch_nodes = np.asarray(json.loads(patch.patch_node_ids), dtype=np.int64)
        x = graph.x.detach().cpu().numpy()
        y = target_vector(graph)
        patch_x = x[patch_nodes]
        patch_y = y[patch_nodes]

        row = {
            "graph_index": int(patch.graph_index),
            "organoid_str": patch.organoid_str,
            "organoid_id": patch.organoid_id,
            "label_uid": patch.label_uid,
            "dataset": patch.dataset,
            "timepoint": patch.timepoint,
            "center_node_id": int(patch.center_node_id),
            "center_node_original_id": int(patch.center_node_original_id),
            "patch_node_ids": patch.patch_node_ids,
            "sampling_stage": patch.sampling_stage,
            "n_cells_pool": int(len(patch_nodes)),
            "n_cells_organoid": int(graph.x.shape[0]),
            "total_surface_area": metadata_value(
                graph,
                ["total_surface_area", "surface_area"],
            ),
            "total_volume": metadata_value(
                graph,
                ["total_volume", "volume"],
            ),
            "center_curvature": float(y[int(patch.center_node_id)]),
            "pool_mean_curvature": float(np.mean(patch_y)),
            "pool_std_curvature": (
                float(np.std(patch_y, ddof=1)) if len(patch_y) > 1 else 0.0
            ),
            "prediction_depth_2": float(patch.center_predicted_curvature),
            "absolute_error_depth_2": float(patch.center_absolute_error),
            "cluster_depth2": int(patch.cluster),
        }

        for marker_index, slug in enumerate(marker_slugs):
            positive = patch_x[:, marker_index] > 0.5
            row[f"n_{slug}_positive"] = int(positive.sum())
            row[f"frac_{slug}_positive"] = float(positive.mean())
            row[f"center_{slug}_positive"] = int(
                x[int(patch.center_node_id), marker_index] > 0.5
            )
            for hop in range(PATCH_RADIUS + 1):
                row[f"hop_{hop}_{slug}_present"] = int(
                    getattr(patch, f"hop_{hop}_marker_{marker_index}_present")
                )

        rows.append(row)

    return pd.DataFrame(rows)

filtered_patch_export_df = build_filtered_patch_export_table(
    accurate_patch_df,
    g_val_raw,
    marker_names,
)
filtered_patch_export_path = (
    TABLES_DIR / "patch_composition_filtered_validation_predictions_clusters.csv"
)
filtered_patch_export_df.to_csv(filtered_patch_export_path, index=False)

print(
    f"Exported {len(filtered_patch_export_df):,} filtered patches -> "
    f"{filtered_patch_export_path}"
)
display(filtered_patch_export_df.head())


## Save tables and settings

Export patch membership, marker fractions, accuracy-selection comparisons and cluster assignments. Record the selected training run/model and analysis parameters alongside the tables.

In [ ]:
patch_df.to_csv(TABLES_DIR / "non_overlapping_patch_table.csv", index=False)
accurate_patch_df.to_csv(TABLES_DIR / "accurate_half_patch_table.csv", index=False)
organoid_sampling_df.to_csv(TABLES_DIR / "organoid_patch_sampling_summary.csv", index=False)
marker_hop_coverage_df.to_csv(TABLES_DIR / "marker_hop_sampling_coverage.csv", index=False)
patch_marker_all_df.to_csv(TABLES_DIR / "patch_marker_values_all.csv", index=False)
patch_marker_accurate_df.to_csv(TABLES_DIR / "patch_marker_values_accurate_half.csv", index=False)
marker_summary_all_df.to_csv(TABLES_DIR / "patch_marker_summary_all.csv", index=False)
marker_summary_accurate_df.to_csv(
    TABLES_DIR / "patch_marker_summary_accurate_half.csv",
    index=False,
)
marker_fraction_change_df.to_csv(
    TABLES_DIR / "patch_marker_fraction_changes_after_filtering.csv",
    index=False,
)
accuracy_filter_summary_df.to_csv(
    TABLES_DIR / "accuracy_filter_summary_by_cluster.csv",
    index=False,
)
cluster_summary_df.to_csv(TABLES_DIR / "validation_cluster_summary.csv", index=False)


In [ ]:
settings=dict(training_run=str(RUN_DIR), model_key=MODEL_KEY, training_settings=run.settings,
    clustering=dict(n_clusters=N_CLUSTERS,embedding=EMBEDDING_VARIANT,seed=CLUSTER_SEED,
                    labels_reordered_by='median true curvature'),
    sampling=dict(radius=PATCH_RADIUS,seed=PATCH_SEED,minimum_marker_hop_coverage=MIN_MARKER_HOP_COVERAGE,
                  coverage_candidate_pool=COVERAGE_CANDIDATE_POOL,max_stage2_patches=MAX_STAGE2_PATCHES),
    accuracy_filter=dict(fraction=ACCURATE_PATCH_FRACTION,minimum=ACCURATE_PATCH_MIN_PER_CLUSTER))
(SAVE_DIR/'settings.json').write_text(json.dumps(settings,indent=2,default=str))
with (SAVE_DIR/'results.pkl').open('wb') as handle:
    pickle.dump(dict(cluster_labels=labels,cluster_label_mapping=old_to_new,patch_df=patch_df,
                    accurate_patch_mask=accurate_patch_mask,marker_summary_all_df=marker_summary_all_df,
                    marker_summary_accurate_df=marker_summary_accurate_df,
                    marker_hop_coverage_df=marker_hop_coverage_df),handle)
print('Saved patch analysis to',SAVE_DIR)